In [218]:
import pandas as pd
import pandas as pd
import json
import os

with open("../config.json") as json_file:
    _LOCAL_CONFIG = json.load(json_file)

derm7pt_folder_path = _LOCAL_CONFIG["derm7pt_folder_path"]
dataset_output_path = _LOCAL_CONFIG["output_folder_path"]

In [219]:
df = pd.read_csv(os.path.join(derm7pt_folder_path, 'meta-raw.csv'))
df_base = pd.read_csv(os.path.join(derm7pt_folder_path, 'metadados.csv'))

In [220]:
df_base.head(2)

,img-id,clinicalDiagnostic,hasElevation,macroBodyRegion,gender
0,NEL025.JPG,basal cell carcinoma,True,ABDOMEN,F
1,NEL027.JPG,basal cell carcinoma,True,NECK,F


In [221]:
df_base.rename(columns={'img-id':'clinical-img-id'}, inplace=True)
df_base.head(2)

,clinical-img-id,clinicalDiagnostic,hasElevation,macroBodyRegion,gender
0,NEL025.JPG,basal cell carcinoma,True,ABDOMEN,F
1,NEL027.JPG,basal cell carcinoma,True,NECK,F


In [222]:
df.head(2)

,case_num,diagnosis,seven_point_score,pigment_network,streaks,pigmentation,regression_structures,dots_and_globules,blue_whitish_veil,vascular_structures,level_of_diagnostic_difficulty,elevation,location,sex,management,clinic,derm,case_id,notes
0,1,basal cell carcinoma,0,absent,absent,absent,absent,absent,absent,arborizing,medium,nodular,abdomen,female,excision,NEL/NEL025.JPG,NEL/Nel026.jpg,NaN,NaN
1,2,basal cell carcinoma,1,absent,absent,absent,absent,irregular,absent,absent,low,palpable,head neck,female,excision,NEL/NEL027.JPG,NEL/Nel028.jpg,NaN,NaN


In [223]:
df['clinic'] = df['clinic'].apply(lambda x: x.split('/')[1])
df = pd.merge(df, df_base, left_on='clinic', right_on='clinical-img-id')
df.columns

Index(['case_num', 'diagnosis', 'seven_point_score', 'pigment_network',
       'streaks', 'pigmentation', 'regression_structures', 'dots_and_globules',
       'blue_whitish_veil', 'vascular_structures',
       'level_of_diagnostic_difficulty', 'elevation', 'location', 'sex',
       'management', 'clinic', 'derm', 'case_id', 'notes', 'clinical-img-id',
       'clinicalDiagnostic', 'hasElevation', 'macroBodyRegion', 'gender'],
      dtype='object')

In [224]:
df['derm'] = df['derm'].apply(lambda x: x.split('/')[1])
df.drop(columns=['streaks', 'pigmentation', 'regression_structures', 'dots_and_globules',
       'blue_whitish_veil', 'vascular_structures', 'seven_point_score',	'pigment_network',
       'level_of_diagnostic_difficulty', 'elevation', 'location', 'sex',
       'management', 'clinic', 'case_id', 'notes'], inplace=True)
df.rename(columns={'case_num' : 'lesion-id'}, inplace=True)


In [225]:
df.head(2)

,lesion-id,diagnosis,derm,clinical-img-id,clinicalDiagnostic,hasElevation,macroBodyRegion,gender
0,1,basal cell carcinoma,Nel026.jpg,NEL025.JPG,basal cell carcinoma,True,ABDOMEN,F
1,2,basal cell carcinoma,Nel028.jpg,NEL027.JPG,basal cell carcinoma,True,NECK,F


In [226]:
df.columns

Index(['lesion-id', 'diagnosis', 'derm', 'clinical-img-id',
       'clinicalDiagnostic', 'hasElevation', 'macroBodyRegion', 'gender'],
      dtype='object')

In [227]:
part1 = df[['lesion-id', 'clinical-img-id', 'clinicalDiagnostic',
       'hasElevation', 'macroBodyRegion', 'gender']].rename(columns={'clinical-img-id': 'img-id'})
part1['img-src'] = 'CLINICAL'
part2 = df[['derm', 'clinicalDiagnostic', 'lesion-id', 
       'hasElevation', 'macroBodyRegion', 'gender']].rename(columns={'derm': 'img-id'})
part2['img-src'] = 'DERMATOSCOPE'

df = pd.concat([part1, part2], ignore_index=True)

df.head()

,lesion-id,img-id,clinicalDiagnostic,hasElevation,macroBodyRegion,gender,img-src
0,1,NEL025.JPG,basal cell carcinoma,True,ABDOMEN,F,CLINICAL
1,2,NEL027.JPG,basal cell carcinoma,True,NECK,F,CLINICAL
2,3,Nel032.jpg,basal cell carcinoma,True,NECK,F,CLINICAL
3,4,NEL034.JPG,basal cell carcinoma,True,LIMBS,M,CLINICAL
4,5,NEL036.JPG,basal cell carcinoma,True,LIMBS,F,CLINICAL


In [228]:
df['clinicalDiagnostic'].unique()

array(['basal cell carcinoma', 'blue nevus', 'clark nevus',
       'combined nevus', 'congenital nevus', 'dermal nevus',
       'dermatofibroma', 'lentigo', 'melanoma (in situ)',
       'melanoma (less than 0.76 mm)', 'melanoma (0.76 to 1.5 mm)',
       'melanoma (more than 1.5 mm)', 'melanoma metastasis', 'melanosis',
       'miscellaneous', 'recurrent nevus', 'reed or spitz nevus',
       'seborrheic keratosis', 'vascular lesion', 'melanoma'],
      dtype=object)

In [229]:
diag_cli_map = {
    'melanoma': 'MEL',
    'melanoma (in situ)': 'MEL',
    'melanoma (less than 0.76 mm)': 'MEL',
    'melanoma (0.76 to 1.5 mm)': 'MEL',
    'melanoma (more than 1.5 mm)': 'MEL', 
    'melanoma metastasis': 'MEL',
    'blue nevus': 'NEVO', 
    'clark nevus': 'NEVO',
    # 'combined nevus': 'NEVO', 
    # 'congenital nevus': 'NEVO', 
    # 'dermal nevus': 'NEVO',
    # 'recurrent nevus': 'NEVO',
    # 'reed or spitz nevus': 'NEVO',
    'basal cell carcinoma': 'CBC',
    'seborrheic keratosis': 'SEBO',
}

df['clinicalDiagnostic'] = df['clinicalDiagnostic'].map(diag_cli_map)
df.head(2)

,lesion-id,img-id,clinicalDiagnostic,hasElevation,macroBodyRegion,gender,img-src
0,1,NEL025.JPG,CBC,True,ABDOMEN,F,CLINICAL
1,2,NEL027.JPG,CBC,True,NECK,F,CLINICAL


In [230]:
diag_map = {
    'MEL' : 'C43',
    'NEVO' : 'D22',
    'CBC' : 'C80',
    'CEC' : 'C44',
    'ACT' : 'L57',
    'SEBO' : 'L82',
}

df['clinicalMacroCID'] = df['clinicalDiagnostic'].map(diag_map)
df.head(2)

,lesion-id,img-id,clinicalDiagnostic,hasElevation,macroBodyRegion,gender,img-src,clinicalMacroCID
0,1,NEL025.JPG,CBC,True,ABDOMEN,F,CLINICAL,C80
1,2,NEL027.JPG,CBC,True,NECK,F,CLINICAL,C80


In [231]:
df['macroBodyRegion'].unique()

array(['ABDOMEN', 'NECK', 'LIMBS', 'BACK', 'CHEST', 'ACRAL', 'BUTTOCKS',
       'AREAS'], dtype=object)

In [232]:
body_map = {
    'ABDOMEN' : 'ABDOME',
    'NECK' : 'PESCOCO',
    'LIMBS' : 'BRACO',
    'BACK' : 'DORSO',
    'CHEST' : 'PEITORAL',
    'ACRAL' : 'PE/MAO',
    'BUTTOCKS' : 'NADEGA',
    'AREAS' : 'OUTROS'
}

df['macroBodyRegion'] = df['macroBodyRegion'].map(body_map)

In [233]:
# df['usePesticide'] = 'I'
# df['familySkinCancerHistory'] = 'I'
# df['familyCancerHistory'] = 'I'
# df['hasItched'] = 'I'
# df['hasGrown'] = 'I'
# df['hasHurt'] = 'I'
# df['hasChanged'] = 'I'
# df['hasBled'] = 'I'
df['histoDiagnostic'] = 'EMPTY'
df['histoMacroCID'] = 'EMPTY'

In [234]:
df['img-id'] = df['img-id'].apply(lambda x: os.path.join(derm7pt_folder_path, 'images', x))

In [235]:
df['img-id'].iloc[52]

'/home/epmagesk/datasets/7-Point-Criteria/images/Adl324.jpg'

In [236]:
df.dropna(inplace=True)

In [237]:
df.head()

,lesion-id,img-id,clinicalDiagnostic,hasElevation,macroBodyRegion,gender,img-src,clinicalMacroCID,histoDiagnostic,histoMacroCID
0,1,/home/epmagesk/datasets/7-Point-Criteria/image...,CBC,True,ABDOME,F,CLINICAL,C80,EMPTY,EMPTY
1,2,/home/epmagesk/datasets/7-Point-Criteria/image...,CBC,True,PESCOCO,F,CLINICAL,C80,EMPTY,EMPTY
2,3,/home/epmagesk/datasets/7-Point-Criteria/image...,CBC,True,PESCOCO,F,CLINICAL,C80,EMPTY,EMPTY
3,4,/home/epmagesk/datasets/7-Point-Criteria/image...,CBC,True,BRACO,M,CLINICAL,C80,EMPTY,EMPTY
4,5,/home/epmagesk/datasets/7-Point-Criteria/image...,CBC,True,BRACO,F,CLINICAL,C80,EMPTY,EMPTY


In [238]:
df = df[df["img-id"].apply(lambda x: os.path.exists(x))]

In [239]:
df.to_csv(os.path.join(dataset_output_path, 'derm7pt_cli_derm_metadata.csv'), index=False)

In [240]:
df['clinicalDiagnostic'].value_counts()

clinicalDiagnostic
NEVO    854
MEL     504
SEBO     90
CBC      84
Name: count, dtype: int64